Hardware:
- Google Colab (Normal)
- NVIDIA Tesla T4

Firstly, we import all code dependencies that will be helpful later on the training process

In [2]:
import pandas as pd
import numpy as np
import os

from google.colab import drive

# Plain Text Generation

In this section we generate a plain text containing all Python scripts included in the datalake. This will be useful in the next section, in order to properly train our gpt-2 fine tuned model

In [3]:
def read_file(path):
  with open(path) as f:
    for line in f.readlines():
      if line[:6] != "<body>":
        return line

  return ""


def read_files(dir_path):
  path_list = os.listdir(dir_path)
  content = ""

  for path in path_list:
    content += read_file(dir_path + "/" + path)

  return content

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
datalake_path = "/content/drive/MyDrive/ulpgc/gpt2/datalake"

In [6]:
text_data = read_files(datalake_path)

Now we split the plain text into train and test data, and store it on drive

In [7]:
train_text = text_data[:int(0.8*len(text_data))]
test_text = text_data[int(0.8*len(text_data)):]

In [8]:
path = "/content/drive/MyDrive/ulpgc/gpt2"

with open(path + "/train_text.txt", "w") as f:
  f.write(train_text)

with open(path + "/test_text.txt", "w") as f:
  f.write(test_text)

# Retraining GPT-2 (Fine tuning)

In this section, we will fine tune GPT-2 using python scripts, all of them obtained via **GitHub Scrapper for RePylot**

In [9]:
from transformers import TextDataset, DataCollatorForLanguageModeling
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments

In [10]:
def load_dataset(file_path, tokenizer, block_size = 128):
    dataset = TextDataset(
        tokenizer = tokenizer,
        file_path = file_path,
        block_size = block_size,
    )
    return dataset


def load_data_collator(tokenizer, mlm = False):
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=mlm,
    )
    return data_collator

In [11]:
def train(train_file_path,model_name,
          output_dir,
          overwrite_output_dir,
          per_device_train_batch_size,
          num_train_epochs,
          save_steps):
  tokenizer = GPT2Tokenizer.from_pretrained(model_name)
  tokenizer.save_pretrained(output_dir)

  train_dataset = load_dataset(train_file_path, tokenizer)
  data_collator = load_data_collator(tokenizer)

  model = GPT2LMHeadModel.from_pretrained(model_name)
  model.save_pretrained(output_dir)

  training_args = TrainingArguments(
          output_dir=output_dir,
          overwrite_output_dir=overwrite_output_dir,
          per_device_train_batch_size=per_device_train_batch_size,
          num_train_epochs=num_train_epochs,
      )

  trainer = Trainer(
          model=model,
          args=training_args,
          data_collator=data_collator,
          train_dataset=train_dataset,
  )

  trainer.train()
  trainer.save_model()

In [12]:
train_file_path = "/content/drive/MyDrive/ulpgc/gpt2/train_text.txt"
model_name = 'gpt2'

output_dir = '/content/drive/MyDrive/ulpgc/gpt2/custom_gpt2'
overwrite_output_dir = True
per_device_train_batch_size = 8
num_train_epochs = 5
save_steps = 5

In [13]:
train(
    train_file_path=train_file_path,
    model_name=model_name,
    output_dir=output_dir,
    overwrite_output_dir=overwrite_output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    save_steps=save_steps
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Step,Training Loss
500,2.349400
1000,2.015500
1500,1.862700
2000,1.777900
2500,1.707900


# Model Evaluation

We can now proceed testing the model we have just trained

In [23]:
def load_model(model_path):
    model = GPT2LMHeadModel.from_pretrained(model_path)
    return model


def load_tokenizer(tokenizer_path):
    tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_path)
    return tokenizer


def generate_text(model_path, sequence, extra_length):
    model = load_model(model_path)
    tokenizer = load_tokenizer(model_path)

    ids = tokenizer.encode(f'{sequence}', return_tensors='pt')
    final_outputs = model.generate(
        ids,
        do_sample=True,
        max_length=len(ids) + extra_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )

    print(tokenizer.decode(final_outputs[0], skip_special_tokens=True))

Feel free to modify `sequence` variable in order to test the model yourself

In [41]:
sequence = "while (t"
generate_text(output_dir, sequence, extra_length=20)

while (tuple(item.count, int))!= len(item.count + 1): yield


If we now compare the obtained result with the original GPT2 model output, we can appreciate how a better response is achieved by our fine tuned transformer. Moreover, the code generated by RePylot is indeed a Python code. Meanwhile, original GPT2 generated code in other language, which possibly doesn't even exist

In [42]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')
set_seed(42)
generator(sequence, max_length=30, num_return_sequences=1)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'while (t < t->size.size) { if (node->flags & PRINT_MODE_INFO) { #ifndef PROC'}]

Other examples are the following

In [47]:
sequence = "for key"

print("RePylot generation:")
generate_text(output_dir, sequence, extra_length=20)

print("\nGPT-2 generation:")
generator(sequence, max_length=30, num_return_sequences=1)

RePylot generation:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


for key in zip(*generate_pair(n, n - key + 1, i), j

GPT-2 generation:


[{'generated_text': 'for keyframes and transitions between keyframes), and a series of options to allow for different styles.\n\nEach of these modes allows you to see'}]

In [49]:
sequence = "from matplotlib"

print("RePylot generation:")
generate_text(output_dir, sequence, extra_length=20)

print("\nGPT-2 generation:")
generator(sequence, max_length=30, num_return_sequences=1)

RePylot generation:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


from matplotlib.animation import pyplot as plt, cvars import pformat as

GPT-2 generation:


[{'generated_text': 'from matplotlib, matplotlib.min.js <~ matplotlib.min.js >) >\n\nNote: Matplot'}]

Note that these results have been obtained fine tuning GPT-2 in only 5 epochs